通过预训练，大模型已经变成了一个“无所不知”的续写机器。但是，如果你对它说：“请帮我写一封感谢信。”它可能不会乖乖写信，而是顺着你的话继续无休止地往下续写：“请帮我写一封感谢信的格式是什么？感谢信怎么写才感人？以下是几篇范文……”

这就是为什么我们需要监督微调（SFT, Supervised Fine-Tuning）。大模型生命周期中的第二大支柱：从“预训练”向“听懂人类指令的助手”蜕变，并攻克 SFT 中最核心的工程难点——Prompt 掩码处理。

# 监督微调（SFT）与 Prompt 掩码机制
1. 预训练（Pre-train）与监督微调（SFT）在数据格式和模型期望上的本质区别是什么？
2. 在计算 SFT 损失时，为什么必须精确地把 Prompt（提示词）部分的 Loss 屏蔽掉（Masking）？

## 从“乱写”到“顺从”——什么是 SFT监督微调？
预训练让模型获得了世界知识（Knowledge），而监督微调（SFT）则是为了规范模型的行为模式（Behavior），让它学会按照“问-答（Instruction-Response）”的对话形式来和人类交互。

在 SFT监督微调 阶段，训练数据不再是互联网上随机抓取的段落，而是由人类专家精心编写或清洗过的高质量指令对：
* Prompt (指令): “计算 $15 \times 6$ 等于多少？”
* Response (回答): “$15 \times 6 = 90$。”

大模型在做 SFT 训练时，网络架构并没有发生任何改变，依然是那个自回归的 Decoder 模型。改变的是数据流和损失函数的计算范围。

## 大模型 SFT 的工程死穴——Prompt 掩码（Prompt Masking）
如果直接把整个 `Prompt` + `Response` 拼接成一条长句子，然后直接用交叉熵去算全句的 Loss，会出现一个极其严重的学术与工程 Bug：**模型会把大量的梯度用在“学习如何预测 Prompt”上**。

为什么不能让模型对 Prompt 算 Loss？
1. **Prompt 是用户输入的，它的逻辑不归模型管**.比如用户输入“我今天心情很差”，这句话是随机的。如果强迫模型去预测“我今天心情____”并计算损失，就是在逼模型去瞎猜用户的下一句话，这会严重混淆和污染模型的参数。
2. 我们只关心模型给出的“回答（Response）”好不好。

因此，在 SFT 训练的工程实现中，我们引入了 Prompt 掩码（通常在 PyTorch 损失函数中使用 -100 表示）。

PyTorch 的 `CrossEntropyLoss` 默认有一个参数叫 `ignore_index=-100`。凡是标签（Label）里被设置成 `-100` 的 Token，在反向传播时完全不计算损失，不产生任何梯度。
* 原始拼接数据：`[Prompt_Token_IDs] + [Response_Token_IDs]`
* 输入给模型的 Input：`[Prompt_Token_IDs] + [Response_Token_IDs]`
* 对应的训练目标 Label：`[-100, -100, ..., -100] + [Response_Token_IDs]`

这样，当模型前向传播处理整句话时，虽然由于因果掩码，Response 依然能看到前面的 Prompt 上下文，但在算 Loss 的那一刻，所有 Prompt 位置对应的 Logits 全被无视，模型只为自己吐出的 Response 的好坏负责！


#### 带 Prompt 掩码的 SFT监督微调 损失计算
我们将模拟大模型在一条对话数据上的 SFT 训练步骤，重点展示如何动态构建 `ignore_index=-100` 的标签矩阵。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ToySFTModel(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.backbone = nn.Linear(d_model, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.token_embedding(idx)
        x = F.gelu(self.backbone(x))
        logits = self.lm_head(x)
        return logits

# --- 模拟监督微调的一步 (SFT Training Step) ---
if __name__ == "__main__":
    torch.manual_seed(7)
    vocab_size = 20 # 假设词表大小为 20
    d_model = 16

    model = ToySFTModel(vocab_size, d_model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

    # 1. 模拟一条 SFT 训练数据
    # 假设分词器处理后，用户提示词 (Prompt) 占了前 3 个 Token，助手的回答 (Response) 占了后 4 个 Token
    prompt_ids = [2,5,9]
    response_ids = [14, 15, 11, 3]

    # 拼接得到模型实际的输入
    input_ids = torch.tensor([prompt_ids + response_ids], dtype=torch.long) # 形状 (1, 7)
    print("【SFT 输入 Token 序列】:", input_ids.tolist()[0])

    # 2. 核心魔法：构建带掩码的 SFT Labels
    # Prompt 部分全部用 -100 填充（不计梯度），Response 部分保留真实 Token ID
    labels = torch.tensor([[-100] * len(prompt_ids) + response_ids], dtype=torch.long)
    print("【初始 SFT Label 序列】:", labels.tolist()[0])

    # 3. 前向传播
    logits = model(input_ids) # 形状: (1, 7, 20)

    # 4. 执行对齐错位 (Shift)
    # 无论是预训练还是 SFT，自回归模型算下一个词的错位逻辑是一样的
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    print("\n--- Shift 错位变换后 ---")
    print("shift_labels 变成:", shift_labels.tolist()[0])

    # 5. 展平张量
    flat_logits = shift_logits.view(-1, vocab_size) # 形状: (6, 20)
    print("flat_logits形状:", flat_logits.shape)
    flat_labels = shift_labels.view(-1) # 形状: (6)
    print("flat_labels形状:", flat_labels.shape)

    # 6. 计算损失：显式指定 ignore_index=-100
    # 此时，展平后的标签里前几个 -100 会被自动跳过，Loss 只由 Response 部分决定
    loss = F.cross_entropy(flat_logits, flat_labels, ignore_index=-100)
    print(f"\n【仅针对 Response 计算的 SFT Loss】: {loss.item():.4f}")

    # 7. 反向传播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print("【SFT 梯度更新成功】模型学会了更好地遵循人类指令！")

1. Shift 后的掩码对齐观察：在上面的输出中，你会发现原本长度为 3 的 Prompt，其在 `shift_labels` 里的 `-100` 数量变成了 2 个（变为 `[-100, -100, 14, 15, 11, 3]`）。请走通以下这个逻辑推导：
    * 在未错位时，输入序列是 `[P1, P2, P3, R1, R2, R3, R4]`
    * 经过错位后，`shift_logits` 的第 2 个位置（索引为 2）代表模型利用 `[P1, P2, P3] 去预测下一个词。
    * 这个预测的目标应该对准谁？它是用户输入的还是模型该回答的？
    * 为什么第 2 个位置的 shift_labels 变成了 14（即 R1）而不是 `-100`？这说明为什么对齐后 `-100` 的数量变少了一个？
2. 多轮对话的扩展思考：在真正的多轮对话 SFT（如 ChatBot 连续和人聊了 3 个回合）数据中，一整条输入数据里会交替出现：
    `Prompt1 + Response1 + Prompt2 + Response2 + Prompt3 + Response3`。如果我们要对这样一条长数据编写 SFT 损失计算的代码，我们的 `labels` 矩阵里的 `-100` 应该怎么分布？（提示：找到所有的 `Prompt` 区域全刷成 `-100`，只保留 `Response` 区域。

仅通过SFT、Prompt 掩码会让模型存在一个致命的缺陷：它只知道什么是“统计学上合理的下文”，而不知道什么是“人类社会里真正好的、安全的回答”。它可能会一本正经地给出错误的垃圾代码，或者由于预训练语料污染而输出带有偏见、甚至危险的内容。
为了解决这一问题，大模型需要迈向 alignment（人类价值观对齐）的关键一步，深度解密人类反馈强化学习（RLHF）的基石——奖励模型（Reward Model）的训练原理与工程实现！

## 人类偏好对齐，手写 Pairwise 排序损失函数
1. 为什么给大模型“直接打分”是一场工程灾难，而“选出更好的那个”才是正确的解法？
2. 在代码上如何徒手实现 RM 训练的核心灵魂——Pairwise 排序损失函数（Margin Loss/Rank Loss）？

#### 为什么我们需要奖励模型（Reward Model）
在 SFT监督微调 阶段，我们给模型的信号是绝对的：这个词对，那个词错。但在现实表达中，很多回答很难说它是绝对的“对”或“错”，而是存在好坏、偏好之分。

人类可以轻易判断出哪种回答更礼貌、更清晰，但我们无法雇佣几万个专家坐在服务器前给大模型每一次训练吐出的词实时打分。因此，我们需要训练另一个小模型作为“数字裁判”，这个模型就叫 奖励模型（Reward Model, RM）。
   * RM 的输入：`Prompt (指令) + Response (回答)`
   * RM 的输出：一个实数标量（如 `1.5` 或 `-2.3`），代表这个回答符合人类偏好的质量得分。

**为什么不能直接让专家给回答打绝对的分数（如 1~10 分）？**
因为人类的打分标准是极其主观且不稳定的。专家 A 心情好可能会给一个回答打 8 分，专家 B 比较严格可能打 5 分。这种不一致的绝对标签喂给神经网络，会导致梯度剧烈震荡，模型根本无法收敛。

但是，人类对“对比”极其敏感且一致。把两个回答放在一起，问专家：“A 和 B 哪个好？” 绝大多数人的判断是高度一致的。这种数据被称为 人类偏好数据（Preference Data），通常以 Pair（对） 的形式呈现.

#### Pairwise 排序损失函数
由于我们只有“谁比谁好”的相对标签，没有绝对得分，因此我们不能用 MSE（均方误差）去逼近绝对值。我们必须使用 Pairwise 排序损失函数（Pairwise Ranking Loss）。

其核心思想是：让奖励模型分别计算 Chosen 回答和 Rejected 回答的得分 $r_\theta(x, y_w)$ 和 $r_\theta(x, y_l)$。我们不关心它们各自具体是多少分，我们只要求它们的“差值”越大越好！数学公式定义如下：
$$\mathcal{L}(\theta) = - \mathbb{E}_{(x, y_w, y_l) \sim D} \left[ \log \sigma \left( r_\theta(x, y_w) - r_\theta(x, y_l) \right) \right]$$
其中 $\sigma$ 是 Sigmoid 函数。让我们拆解它的物理意义：
1. 当 $r_\theta(x, y_w)$ 远大于 $r_\theta(x, y_l)$ 时，差值很大，$\sigma(\text{差值}) \to 1$，$\log(1) = 0$，此时 Loss 接近 0，模型不需要调整。
2. 当 $r_\theta(x, y_w)$ 小于 $r_\theta(x, y_l)$ 时（即模型把高分错给了差的回答），差值小于 0，$\sigma(\text{差值}) \to 0$，$-\log(\to 0)$ 会飙升到无穷大！产生极其恐怖的 Loss，强迫模型剧烈修正权重。

用 PyTorch 模拟一条偏好数据在奖励模型里的数据流与 Margin Loss 计算:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 构建一个奖励模型 (Reward Model)
# 它的架构与普通的 Decoder 模型极其相似，唯一的区别是 LM Head 映射到的 vocab_size 是 1
# 也就是说，它在句子结束时，只吐出一个代表总分的标量数值
class ToyRewardModel(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.backbone = nn.Linear(d_model, d_model)
        # 奖励头 (Value Head)：把特征维度映射为 1 个标量分值
        self.v_head = nn.Linear(d_model, 1, bias=False)

    def forward(self, idx):
        # idx 形状: (batch_size, seq_len)
        x = self.token_embedding(idx)
        x = F.gelu(self.backbone(x))

        # 拿到整句话最后一个 Token 位置的表征（它凝聚了全句和上下文的所有信息）
        last_token_feat = x[:, -1, :] # 形状: (batch_size, d_model)

        score = self.v_head(last_token_feat) # 形状: (batch_size, 1)
        return score.squeeze(-1) # 压平输出: (batch_size,)

# --- 模拟奖励模型的单步训练 ---
if __name__ == "__main__":
    torch.manual_seed(42)
    vocab_size = 30
    d_model = 16

    reward_model = ToyRewardModel(vocab_size, d_model)
    optimizer = torch.optim.AdamW(reward_model.parameters(), lr=0.01)

    # 1. 模拟一条人类偏好数据 (Pairwise Data)
    prompt = [1, 2, 3]
    chosen_resp = [4, 5, 6]    # 专家认为好的回答
    rejected_resp = [7, 8]     # 专家认为差的回答

    # 2. 将 Prompt 与两条回答分别拼接，组合成喂给 RM 的两条独立序列
    chosen_sequence = torch.tensor([prompt + chosen_resp], dtype=torch.long)   # 形状: (1, 6)
    rejected_sequence = torch.tensor([prompt + rejected_resp], dtype=torch.long) # 形状: (1, 5)

    # 3. 前向传播，分别计算两个完整序列的得分
    # 注意：在真实的工业工程中，为了压榨 GPU 算力，通常会把 chosen 和 rejected 打包成一个 Batch 传入
    # 这里我们分开写，让你看得更通透
    chosen_score = reward_model(chosen_sequence)     # 得到 r(x, y_w)
    rejected_score = reward_model(rejected_sequence) # 得到 r(x, y_l)

    print(f"【模型预测】Chosen 回答得分: {chosen_score.item():.4f}")
    print(f"【模型预测】Rejected 回答得分: {rejected_score.item():.4f}")

    # 4. 核心工程实现：Pairwise Ranking Loss
    # 我们希望 chosen_score - rejected_score 越大越好
    score_diff = chosen_score - rejected_score

    # 用数学公式：Loss = -log(sigmoid(chosen_score - rejected_score))
    loss = -torch.log(torch.sigmoid(score_diff)).mean()
    print(f"\n【计算出的 Pairwise Ranking Loss】: {loss.item():.4f}")

    # 5. 反向传播更新裁判模型
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print("\n【RM 梯度更新成功】数字裁判变得更加严明，更加懂人类的喜恶了！")

1. 观察反向传播的物理纠偏：在上面的代码运行结果中，你会发现因为模型权重是随机初始化的，一开始它可能会给出荒谬的打分（例如把 Rejected 差回答打的分数比 Chosen 好回答还要高）。请在脑海里或者纸上推导一下：
    * 在这种“判错案”的情况下，`score_diff` 是正数还是负数？
    * 此时经过 `torch.sigmoid` 之后的概率值是大于 0.5 还是远小于 0.5？
    * 这会导致算出来的 Loss 是很大还是根部没有？它所产生的梯度会如何“猛烈抽打” `v_head` 的权重，使其在下一个 Step 迅速扭转局面？
2. Margin（边界金牌加分）的进阶思考：在工业界（如最顶尖的 Llama 3 或 Anthropic Claude 训练中），人们往往会给排序损失函数加上一个常量边界 $\gamma$ (Margin)，公式变成 $-\log \sigma(r(x, y_w) - r(x, y_l) - \gamma)$。如果两个回答的质量差距非常明显（比如一个是完美解答，另一个是完全胡说八道），我们通常会将 $\gamma$ 调大。请问引入这个 Margin 边界的工程目的是什么？如果不加 $\gamma$，当模型好不容易把 Chosen 训练得比 Rejected 高出了一小分（比如 0.1 分）时，模型就会摆烂停滞；而加上了 $\gamma=2.0$ 会强制要求模型怎么做？这对于提高奖励模型的判别敏锐度有什么决定性的帮助？